# Exact-set and count-based analysis of mouse TRA/TRB repertoires

This notebook contains the main analysis for the BALB/c to C57BL/6
allotransplantation study. Each execution handles one biological stratum and combines
exact aaV set analysis with edgeR and a pooled-count Fisher test.

The original notebook version used as the reconstruction reference is commit
`6678766a0e8913ec2feacb3efb2daebd6a27eda4`, blob
`043ec6bf8227b72ea3e3bfc6683aaac1622fe341`.

Two evidence streams are used:

1. exact `aaV` clonotype sets, defined by `(CDR3 amino-acid sequence, V segment)`,
   for group intersections and V-segment localization;
2. count-based inference for the g1 versus g5+g6 contrast, using mouse-pooled edgeR
   where required and a pooled-count Fisher test.

TRA/TRB repertoire similarity and mouse-level UMI correlation are also retained as
numerical tables. Figures produced during execution are saved in the directory for the
current stratum.

Treatment groups follow the study metadata: g1, intact BALB/c recipients; g2, C57BL/6
donors; g3, syngeneic bone-marrow transplantation; g4, conditioning control; g5,
allogeneic bone-marrow transplantation; and g6, allogeneic bone-marrow transplantation
with donor thymus.

The notebook does not contain a fixed biological conclusion. Rankings are calculated
separately for each stratum, with concordant, single-method and descriptive evidence
reported separately.

The default stratum is `all_combined`: CD4 and CD8 samples from thymus and spleen.
This is the union of the sampled subsets, not a CD4/CD8 double-positive population.
Exact-set membership uses the original combined sample scope; count-based inference
pools repeated samples within mouse. Spike-in mode runs the same notebook for the
baseline and each count perturbation; see `docs/spike_in.md`.


## Runtime parameters and output contract


In [ ]:
from pathlib import Path
import ast
import itertools
import json
import math
import os
import re
import sys

REPOSITORY_ROOT = Path(os.environ.get("MICE_TCR_REPO", Path.cwd())).expanduser().resolve()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.runtime import configure_repseq_parallelism, configure_runtime
from src.figures import CORAL, PURPLE, apply_article_style
from src.article_figures import (
    ARTICLE_FIGURES, ARTICLE_FONTS,
    article_canvas, article_dpi, article_name, panel_bbox,
    plot_dumbbell, plot_retention, plot_localization, plot_subsets, plot_edger, plot_fisher, plot_fisher_v,
)
from src.structural_sampling import _stable_sample_seed
from src.v_thresholds import FIXED_THRESHOLDS, fit_mouse_dropout_thresholds
from src.repertoire_depth import depth_check, mds_points, plot_depth_check, plot_mds

N_WORKERS = configure_runtime()
ACTIVE_CHAIN = "TCR"

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from matplotlib.patches import Circle
from matplotlib_venn import venn2, venn3
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from repseq import clonosets as cl
from repseq import clustering
from repseq import diffexp
from repseq import intersections
from repseq import io as repseqio
from repseq import logo
from repseq import mixcr as mx
from repseq import slurm
from repseq import stats
from repseq import vdjtools
from repseq import clone_filter as clf

configure_repseq_parallelism(N_WORKERS)

from src.strata import STRATUM_DEFINITIONS, STRATUM_LABELS
from src.spike_in import (
    apply_spike_to_tables, combine_count_tables, snapshot_tables,
    load_snapshot_tables, write_case_metrics, validate_case_name,
)

STRATUM_CONFIG = {
    key: {**definition, "label": STRATUM_LABELS[key]}
    for key, definition in STRATUM_DEFINITIONS.items()
}

STRATUM = os.environ.get("MICE_TCR_STRATUM", "all_combined").strip().lower()
if STRATUM not in STRATUM_CONFIG:
    raise ValueError(
        f"Unknown MICE_TCR_STRATUM={STRATUM!r}. "
        f"Expected one of: {', '.join(STRATUM_CONFIG)}"
    )
STRATUM_DEFINITION = STRATUM_CONFIG[STRATUM]
STRATUM_LABEL = STRATUM_DEFINITION["label"]
FIGURE_STRATUM_SUFFIX = "" if STRATUM == "all_combined" else f" | {STRATUM_LABEL}"
POOL_WITHIN_MOUSE = STRATUM_DEFINITION["pool_within_mouse"]

RUN_MODE = os.environ.get("MICE_TCR_MODE", "standard")
SPIKE_CASE = os.environ.get("MICE_TCR_SPIKE_CASE", "baseline")
SPIKE_RUN_ID = os.environ.get("MICE_TCR_SPIKE_RUN_ID", "interactive")
SPIKE_CHAIN = os.environ.get("MICE_TCR_SPIKE_CHAIN", "TRA")
SPIKE_FRACTION = float(os.environ.get("MICE_TCR_SPIKE_FRACTION", "0"))
SPIKE_SELECTION = None
SPIKE_DOSE_TABLE = None
V_THRESHOLD_MODE = os.environ.get("MICE_TCR_V_THRESHOLDS", "adaptive") if RUN_MODE == "standard" else "fixed"
V_THRESHOLD_RECORDS = []
APPROACH = "01_set_count"
if RUN_MODE == "v-inputs":
    APPROACH = "01_v_update_inputs"
FIGURE_ROOT = REPOSITORY_ROOT / "figures" / APPROACH
RESULT_ROOT = REPOSITORY_ROOT / "results" / APPROACH
FIGURE_DIR = FIGURE_ROOT / STRATUM
RESULT_DIR = RESULT_ROOT / STRATUM
if RUN_MODE == "spike-in":
    if STRATUM != "all_combined":
        raise ValueError("Spike-in requires all_combined.")
    if SPIKE_CHAIN not in {"TRA", "TRB"}:
        raise ValueError("The spike-in chain must be TRA or TRB.")
    validate_case_name(SPIKE_RUN_ID)
    validate_case_name(SPIKE_CASE)
    APPROACH = "01_spike_in"
    FIGURE_ROOT = REPOSITORY_ROOT / "figures" / APPROACH / SPIKE_RUN_ID
    RESULT_ROOT = REPOSITORY_ROOT / "results" / APPROACH / SPIKE_RUN_ID
    FIGURE_DIR = FIGURE_ROOT / SPIKE_CASE
    RESULT_DIR = RESULT_ROOT / SPIKE_CASE
    if SPIKE_CASE != "baseline":
        SPIKE_SELECTION = json.loads((RESULT_ROOT / "selection.json").read_text())
    STRATUM_LABEL += f" | spike-in {SPIKE_CASE} ({100 * SPIKE_FRACTION:g}% added UMI)"
elif RUN_MODE not in {"standard", "v-inputs"}:
    raise ValueError(f"Unknown run mode: {RUN_MODE}")
if RUN_MODE != "v-inputs":
    for chain in ("TRA", "TRB"):
        (FIGURE_DIR / chain).mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Biological stratum: {STRATUM} | {STRATUM_LABEL}")
print(f"Mouse-level pooling for count inference: {POOL_WITHIN_MOUSE}")
print(f"Figure directory: {FIGURE_DIR}")
print(f"Result directory: {RESULT_DIR}")

# record software and source identity with every executed analysis
import hashlib
import importlib.metadata
import platform
import subprocess
try:
    revision = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_ROOT, text=True,
        stderr=subprocess.DEVNULL,
    ).strip()
except (OSError, subprocess.CalledProcessError):
    revision = None
versions = {}
for package in ("numpy", "pandas", "scipy", "statsmodels", "matplotlib", "repseq", "rpy2"):
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = "not recorded"
provenance = {
    "git_revision": revision, "python": platform.python_version(), "packages": versions,
    "stratum": STRATUM, "mode": RUN_MODE,
    "v_threshold_mode": V_THRESHOLD_MODE,
    "notebook_sha256": hashlib.sha256((REPOSITORY_ROOT / "venn_original.ipynb").read_bytes()).hexdigest(),
    "source_sha256": {
        str(path.relative_to(REPOSITORY_ROOT)): hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted((REPOSITORY_ROOT / "src").glob("*.py"))
    },
}
(RESULT_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")

In [ ]:
_ORIGINAL_PLT_SAVEFIG = plt.savefig
_ORIGINAL_PLT_SHOW = plt.show
_PLOT_SEQUENCE = 0
PLOT_RECORDS = []


def _slugify_figure_name(value):
    value = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()
    return value[:100] or "untitled_figure"


def _figure_title(fig):
    if fig._suptitle is not None and fig._suptitle.get_text().strip():
        return fig._suptitle.get_text().strip()
    titles = [axis.get_title().strip() for axis in fig.axes if axis.get_title().strip()]
    return "; ".join(titles) or "TCR repertoire analysis"


def _figure_chain(fig, requested_path=None):
    requested_name = Path(str(requested_path)).name if requested_path is not None else ""
    explicit = f"{_figure_title(fig)} {requested_name}".upper()
    has_tra = re.search(r"(^|[^A-Z0-9])TRA(?:V|[^A-Z0-9]|$)", explicit) is not None
    has_trb = re.search(r"(^|[^A-Z0-9])TRB(?:V|[^A-Z0-9]|$)", explicit) is not None
    if has_tra and not has_trb:
        return "TRA"
    if has_trb and not has_tra:
        return "TRB"
    if ACTIVE_CHAIN in {"TRA", "TRB"}:
        return ACTIVE_CHAIN
    return None


def _central_figure_path(requested_path, fig):
    global _PLOT_SEQUENCE
    _PLOT_SEQUENCE += 1
    chain = _figure_chain(fig, requested_path)
    output_dir = FIGURE_DIR / chain if chain else FIGURE_DIR
    filename = Path(str(requested_path)).with_suffix(".png").name if requested_path else None
    article_names = {article_name(chain, panel) for panel in ARTICLE_FIGURES} if chain else set()
    if filename not in article_names or RUN_MODE == "spike-in":
        output_dir = output_dir / "surplus"
    output_dir.mkdir(parents=True, exist_ok=True)
    if requested_path is None:
        stem = f"{_PLOT_SEQUENCE:03d}_{_slugify_figure_name(_figure_title(fig))}"
        return output_dir / f"{stem}.png"
    return output_dir / filename


def _record_figure(fig, png_path, origin):
    key = str(png_path.resolve())
    if key not in {record["png"] for record in PLOT_RECORDS}:
        PLOT_RECORDS.append(
            {
                "stratum": STRATUM,
                "chain": _figure_chain(fig, png_path) or "",
                "title": _figure_title(fig),
                "origin": origin,
                "png": key,
            }
        )
    saved = set(getattr(fig, "_article_saved_png", set()))
    saved.add(key)
    fig._article_saved_png = saved


def _save_figure_png(fig, requested_path=None, origin="inline", **kwargs):
    png_path = _central_figure_path(requested_path, fig)
    options = dict(kwargs)
    options.pop("format", None)
    options.setdefault("dpi", 300)
    options.setdefault("bbox_inches", "tight")
    if origin == "article" or png_path.name.endswith("_stability.png"):
        from src.publication_figures import save_publication_figure
        save_publication_figure(fig, png_path)
    else:
        fig.savefig(png_path, **options)
    _record_figure(fig, png_path, origin)
    return png_path


def _article_savefig(path, *args, **kwargs):
    if args:
        raise TypeError("Positional savefig arguments are not supported by this notebook.")
    saved_path = _save_figure_png(plt.gcf(), path, origin="explicit", **kwargs)
    print(f"Saved figure: {saved_path.relative_to(REPOSITORY_ROOT)}")
    return None


def _persist_open_figures(origin="inline"):
    for figure_number in list(plt.get_fignums()):
        fig = plt.figure(figure_number)
        if not getattr(fig, "_article_saved_png", set()):
            _save_figure_png(fig, origin=origin)


def _article_show(*args, **kwargs):
    _persist_open_figures(origin="displayed")
    return _ORIGINAL_PLT_SHOW(*args, **kwargs)


def _render_article(fig, chain, panel):
    filename = article_name(chain, panel)
    _save_figure_png(fig, filename, origin="article", dpi=article_dpi(filename), bbox_inches=None)
    plt.show()
    plt.close(fig)


plt.savefig = _article_savefig
plt.show = _article_show

apply_article_style()
sns.set_theme(style="white", context="notebook")
plt.rcParams.update(
    {
        "font.sans-serif": ARTICLE_FONTS,
        "axes.grid": False,
        "axes.titleweight": "normal",
        "savefig.facecolor": "white",
    }
)

In [ ]:
WORKING_DIR = Path(
    os.environ.get("MICE_TCR_WORKING_DIR", "/projects/mice_transplant_2025/test_run")
).expanduser().resolve()
METADATA_FILENAME = Path(
    os.environ.get(
        "MICE_TCR_METADATA_CSV",
        "/projects/mice_transplant_2025/metadata_mice_transplant.csv",
    )
).expanduser().resolve()
CLONOSET_INDEX_FILENAME = Path(
    os.environ.get(
        "MICE_TCR_CLONOSET_INDEX",
        WORKING_DIR / "clonosets_mice_transplant_2025_df.csv",
    )
).expanduser().resolve()

for required_path in (METADATA_FILENAME, CLONOSET_INDEX_FILENAME):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required input file not found: {required_path}")

metadata_all = pd.read_csv(METADATA_FILENAME)
required_metadata_columns = {
    "chain", "sample_no", "sample_id", "group_no", "mouse_no", "source", "subtype"
}
missing_metadata = required_metadata_columns.difference(metadata_all.columns)
if missing_metadata:
    raise ValueError(f"Metadata columns are missing: {sorted(missing_metadata)}")


def _identifier(value):
    if pd.isna(value):
        return ""
    try:
        numeric = float(value)
    except (TypeError, ValueError):
        return str(value).strip()
    return str(int(numeric)) if numeric.is_integer() else str(value).strip()


metadata_all = metadata_all.copy()
metadata_all["sample_id_old"] = (
    metadata_all["chain"].astype(str).str.strip()
    + "-"
    + metadata_all["sample_no"].map(_identifier)
)
metadata_all = metadata_all.drop(columns="chain")

clonosets = pd.read_csv(CLONOSET_INDEX_FILENAME)
required_index_columns = {"sample_id", "filename", "chain"}
missing_index = required_index_columns.difference(clonosets.columns)
if missing_index:
    raise ValueError(f"Clonoset-index columns are missing: {sorted(missing_index)}")

clonosets = clonosets.rename(columns={"sample_id": "sample_id_old"})
clonosets = clonosets.merge(
    metadata_all,
    on="sample_id_old",
    how="left",
    validate="many_to_one",
    indicator=True,
)
unmatched = clonosets.loc[clonosets["_merge"].ne("both"), "sample_id_old"].tolist()
if unmatched:
    raise ValueError(
        "Clonoset rows could not be matched to metadata by the historical "
        "chain/sample-number key: " + ", ".join(map(str, unmatched[:10]))
    )
clonosets = clonosets.drop(columns=["sample_id_old", "_merge"])


def _resolve_clonoset_path(value):
    path = Path(str(value)).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates.extend(
            [CLONOSET_INDEX_FILENAME.parent / path, WORKING_DIR / "mixcr" / path]
        )
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate.resolve())
    return str(candidates[-1].resolve())


clonosets["filename"] = clonosets["filename"].map(_resolve_clonoset_path)
missing_clonosets = [path for path in clonosets["filename"] if not Path(path).is_file()]
if missing_clonosets:
    raise FileNotFoundError(
        "MiXCR clonotype exports referenced by the index were not found: "
        + "; ".join(missing_clonosets[:10])
    )

classification_text = (
    clonosets["subtype"].fillna("").astype(str).str.lower()
    + " "
    + clonosets["sample_id"].fillna("").astype(str).str.lower()
)
source_text = (
    clonosets["source"].fillna("").astype(str).str.lower()
    + " "
    + clonosets["sample_id"].fillna("").astype(str).str.lower()
)
clonosets["_cell_subset"] = np.select(
    [
        classification_text.str.contains(r"(?:^|[^a-z0-9])cd4(?:[^a-z0-9]|$)", regex=True),
        classification_text.str.contains(r"(?:^|[^a-z0-9])cd8(?:[^a-z0-9]|$)", regex=True),
    ],
    ["cd4", "cd8"],
    default="unresolved",
)
clonosets["_tissue"] = np.select(
    [source_text.str.contains("spleen"), source_text.str.contains("thym")],
    ["spleen", "thymus"],
    default="unresolved",
)
unresolved = clonosets[
    clonosets["_cell_subset"].eq("unresolved") | clonosets["_tissue"].eq("unresolved")
]["sample_id"].unique()
if len(unresolved):
    raise ValueError(
        "Every sample must map unambiguously to CD4/CD8 and thymus/spleen. "
        f"Unresolved samples: {', '.join(map(str, unresolved[:10]))}"
    )

clonosets = clonosets[
    clonosets["_cell_subset"].isin(STRATUM_DEFINITION["cell_subsets"])
    & clonosets["_tissue"].isin(STRATUM_DEFINITION["tissues"])
].copy()
if clonosets.empty:
    raise ValueError(f"No samples remain for biological stratum {STRATUM!r}.")

clonosets["chain"] = clonosets["chain"].astype(str).str.upper().str.strip()
clonosets["group_no"] = clonosets["group_no"].astype(str).str.lower().str.strip()
clonosets = clonosets[
    ["sample_id"] + [column for column in clonosets.columns if column != "sample_id"]
].sort_values("sample_id").reset_index(drop=True)

metadata = clonosets.drop_duplicates("sample_id").copy()
metadata["mouse_id"] = (
    metadata["group_no"].astype(str)
    + "_m"
    + metadata["mouse_no"].map(_identifier)
)

historical_exclusions = {
    "g1_m3_thymus_cd8_80_alpha",
    "g5_m1_spleen_cd4_93_alpha",
    "g5_m4_thymus_cd8_100_alpha",
}

def _group_samples(chain, group_no):
    selected = clonosets[
        clonosets["chain"].eq(chain) & clonosets["group_no"].eq(group_no)
    ].copy()
    if chain == "TRA":
        selected = selected[~selected["sample_id"].isin(historical_exclusions)]
    return selected.sort_values("umi_func")


for chain_name in ("TRA", "TRB"):
    for group_name in ("g1", "g2", "g3", "g4", "g5", "g6"):
        globals()[f"clonosets_{chain_name.lower()}_{group_name}"] = _group_samples(
            chain_name, group_name
        )

clonosets_tra_g6_allo = clonosets_tra_g6[
    ~clonosets_tra_g6["source"].astype(str).str.lower().eq("thymus")
].copy()
clonosets_tra_g6_auto = clonosets_tra_g6[
    ~clonosets_tra_g6["source"].astype(str).str.lower().eq("allothymus")
].copy()
clonosets_trb_g6_allo = clonosets_trb_g6[
    ~clonosets_trb_g6["source"].astype(str).str.lower().eq("thymus")
].copy()
clonosets_trb_g6_auto = clonosets_trb_g6[
    ~clonosets_trb_g6["source"].astype(str).str.lower().eq("allothymus")
].copy()

_REPSEQ_COUNT_TABLE = intersections.count_table


def _safe_count_table(sample_index, *args, **kwargs):
    if sample_index is None or len(sample_index) == 0:
        print("No samples were available for this group; returning an empty count table.")
        return pd.DataFrame(index=pd.Index([], dtype="object"))
    return _REPSEQ_COUNT_TABLE(sample_index, *args, **kwargs)


intersections.count_table = _safe_count_table

sample_audit = (
    metadata.groupby(["chain", "group_no"], observed=True)
    .agg(samples=("sample_id", "nunique"), mice=("mouse_id", "nunique"))
    .reset_index()
)
sample_audit.to_csv(RESULT_DIR / "selected_samples_by_chain_and_group.csv", index=False)
display(sample_audit)

DOWNSAMPLE_DEPTH = int(os.environ.get("MICE_TCR_DOWNSAMPLE_UMI", "15000"))
DOWNSAMPLE_SEED = int(os.environ.get("MICE_TCR_DOWNSAMPLE_SEED", "1031"))
from src.v_stability import (
    RAREFACTION_SEEDS, rarefaction_profiles, representative_seed, write_stability,
)
REPRESENTATIVE_SEED_STRATA = {"all_combined", "cd4_combined", "cd8_combined"}
FUNCTIONAL_UMI_FILTER = clf.Filter(
    functionality="f",
    by_umi=True,
    recount_fractions=True,
)


def downsample_group_tables(
    tables,
    depth=DOWNSAMPLE_DEPTH,
    seed=DOWNSAMPLE_SEED,
    eligible_by_group=None,
):
    from src.structural_sampling import downsample_group_tables as rarefy_groups

    return rarefy_groups(tables, depth=depth, seed=seed, eligible_by_group=eligible_by_group)


## TRA exact-set analysis


In [ ]:
ACTIVE_CHAIN = "TRA"
baseline_eligible_samples = None
if RUN_MODE == "spike-in" and SPIKE_CASE != "baseline":
    baseline_chain_tables = load_snapshot_tables(RESULT_ROOT / "baseline", "TRA", metadata)
    baseline_eligible_samples = {
        group: {
            str(sample_id)
            for sample_id in table.columns
            if int(table[sample_id].sum()) >= DOWNSAMPLE_DEPTH
        }
        for group, table in baseline_chain_tables.items()
    }
    raw_chain_tables = {group: table.copy() for group, table in baseline_chain_tables.items()}
else:
    raw_chain_tables = {
        group: intersections.count_table(
            globals()[f"clonosets_tra_{group}"],
            cl_filter=FUNCTIONAL_UMI_FILTER,
            overlap_type="aaV",
            mismatches=0,
        )
        for group in ("g1", "g2", "g3", "g4", "g5", "g6")
        if RUN_MODE != "v-inputs" or group != "g3"
    }
    if RUN_MODE == "spike-in":
        snapshot_tables(raw_chain_tables, RESULT_DIR, "TRA", metadata)

if SPIKE_SELECTION is not None and SPIKE_CHAIN == "TRA":
    raw_chain_tables, SPIKE_DOSE_TABLE = apply_spike_to_tables(
        raw_chain_tables, SPIKE_SELECTION, SPIKE_FRACTION
    )
    SPIKE_DOSE_TABLE.to_csv(RESULT_DIR / "spike_dose_by_sample.csv", index=False)

ACTIVE_STRUCTURAL_SEED = DOWNSAMPLE_SEED
if RUN_MODE == "v-inputs":
    baseline_results = Path(os.environ["MICE_TCR_V_BASELINE_RESULTS"])
    baseline_summary = json.loads((baseline_results / "run_summary.json").read_text())
    ACTIVE_STRUCTURAL_SEED = int(baseline_summary["structural_downsample_seed"])
    DOWNSAMPLE_DEPTH = int(baseline_summary["structural_downsample_umi"])
TRA_RAREFACTIONS = None
if RUN_MODE == "v-inputs" or V_THRESHOLD_MODE == "adaptive":
    raw_chain_tables = {group: table.sort_index() for group, table in raw_chain_tables.items()}

if STRATUM in REPRESENTATIVE_SEED_STRATA and RUN_MODE == "standard":
    TRA_RAREFACTIONS = rarefaction_profiles(raw_chain_tables, DOWNSAMPLE_DEPTH)
    selected_seed, seed_summary = representative_seed(TRA_RAREFACTIONS[1])
    if selected_seed is not None:
        ACTIVE_STRUCTURAL_SEED = selected_seed
    seed_summary.to_csv(RESULT_DIR / "tra_rarefaction_seed_summary.csv", index=False)
    (RESULT_DIR / "representative_downsampling_seed.json").write_text(
        json.dumps(
            {
                "selected_seed": int(ACTIVE_STRUCTURAL_SEED),
                "n_reruns": len(RAREFACTION_SEEDS),
                "selection_status": "adaptive_medoid" if selected_seed is not None else "no_valid_cutoffs",
                "selection_metric": "mean Jaccard distance between complete adaptive TRAV shortlists",
                "tie_breaker": "lower seed",
                "spike_in_used_for_selection": False,
            },
            indent=2,
        ) + "\n",
        encoding="utf-8",
    )

chain_tables, tra_depth_audit = downsample_group_tables(
    raw_chain_tables,
    depth=DOWNSAMPLE_DEPTH,
    seed=ACTIVE_STRUCTURAL_SEED,
    eligible_by_group=baseline_eligible_samples,
)
tra_depth_audit.to_csv(RESULT_DIR / "tra_structural_depth_audit.csv", index=False)

for group in raw_chain_tables:
    globals()[f"raw_ct_{group}_aaV_tra"] = raw_chain_tables[group]
    globals()[f"ct_{group}_aaV_tra"] = chain_tables[group]

if RUN_MODE == "v-inputs":
    metadata.to_csv(RESULT_DIR / "mouse_metadata.csv", index=False)
    snapshot_tables(raw_chain_tables, RESULT_DIR / "raw", ACTIVE_CHAIN, metadata)
    snapshot_tables(chain_tables, RESULT_DIR / "structural", ACTIVE_CHAIN, metadata)
    (RESULT_DIR / "v_input_seeds.json").write_text(json.dumps({
        "baseline_results": str(baseline_results),
        "structural_seed": ACTIVE_STRUCTURAL_SEED, "calibration_seed": DOWNSAMPLE_SEED,
        "depth": DOWNSAMPLE_DEPTH,
    }, indent=2) + "\n")

print(
    f"TRA structural sets use functional clonotypes downsampled to "
    f"{DOWNSAMPLE_DEPTH:,} UMI per eligible library with seed {ACTIVE_STRUCTURAL_SEED}"
)

In [ ]:
def plot_double_triple_venn_like(
    center_set,
    left_top_set,
    left_bottom_set,
    right_top_set,
    right_bottom_set,
    center_label,
    left_top_label,
    left_bottom_label,
    right_top_label,
    right_bottom_label,
    title=None,
    save_path=None,
):
    """Draw the five-set layout used for the article background-subtraction figure."""
    center_set = set(center_set)
    left_top_set = set(left_top_set)
    left_bottom_set = set(left_bottom_set)
    right_top_set = set(right_top_set)
    right_bottom_set = set(right_bottom_set)

    left_top_only = left_top_set - (center_set | left_bottom_set)
    left_bottom_only = left_bottom_set - (center_set | left_top_set)
    left_top_left_bottom_only = (left_top_set & left_bottom_set) - center_set
    center_left_top_only = (center_set & left_top_set) - left_bottom_set
    center_left_bottom_only = (center_set & left_bottom_set) - left_top_set
    left_triple = center_set & left_top_set & left_bottom_set

    right_top_only = right_top_set - (center_set | right_bottom_set)
    right_bottom_only = right_bottom_set - (center_set | right_top_set)
    right_top_right_bottom_only = (right_top_set & right_bottom_set) - center_set
    center_right_top_only = (center_set & right_top_set) - right_bottom_set
    center_right_bottom_only = (center_set & right_bottom_set) - right_top_set
    right_triple = center_set & right_top_set & right_bottom_set

    center_only_after_all = center_set - (
        left_top_set | left_bottom_set | right_top_set | right_bottom_set
    )

    fig = article_canvas("overlap")
    ax = fig.add_axes((0.043, 0.062, 0.914, 0.842))
    ax.set_aspect("equal")
    ax.axis("off")

    colors = {
        left_top_label: "#f8837a",
        left_bottom_label: "#a9589b",
        center_label: "#ffcf9b",
        right_top_label: "#ffac7c",
        right_bottom_label: "#865fa0",
    }
    circles = {
        left_top_label: {"xy": (-2.12, 0.80), "r": 1.435, "set": left_top_set},
        left_bottom_label: {"xy": (-2.12, -0.80), "r": 1.435, "set": left_bottom_set},
        center_label: {"xy": (0.0, 0.0), "r": 1.435, "set": center_set},
        right_top_label: {"xy": (2.12, 0.80), "r": 1.435, "set": right_top_set},
        right_bottom_label: {"xy": (2.12, -0.80), "r": 1.435, "set": right_bottom_set},
    }
    for label, params in circles.items():
        ax.add_patch(
            Circle(
                params["xy"],
                params["r"],
                facecolor=to_rgba(colors[label], 0.42),
                edgecolor="#a19b99",
                linewidth=0.8,
                zorder=1,
            )
        )

    label_positions = {
        left_top_label: (-3.16, 1.51),
        left_bottom_label: (-3.16, -1.62),
        center_label: (0.0, -1.25),
        right_top_label: (3.16, 1.51),
        right_bottom_label: (3.16, -1.62),
    }
    for label, (x, y) in label_positions.items():
        ax.text(
            x,
            y,
            f"{label}\n{len(circles[label]['set']):,}",
            ha="center",
            va="center",
            fontsize=11.5,
            fontweight="bold",
            color="#222222",
            zorder=5,
        )

    def put_count(x, y, values):
        ax.text(
            x,
            y,
            f"{len(values):,}",
            ha="center",
            va="center",
            fontsize=9.8,
            color="#222222",
            bbox=dict(
                boxstyle="square,pad=0.29",
                facecolor="white",
                edgecolor="#EEEEEE",
                linewidth=0,
                alpha=0.88,
            ),
            zorder=10,
        )

    put_count(-2.70, 0.77, left_top_only)
    put_count(-2.70, -0.90, left_bottom_only)
    put_count(-2.12, 0.00, left_top_left_bottom_only)
    put_count(-1.12, 0.48, center_left_top_only)
    put_count(-1.12, -0.48, center_left_bottom_only)
    put_count(-1.38, 0.00, left_triple)
    put_count(0.00, 0.00, center_only_after_all)
    put_count(1.38, 0.00, right_triple)
    put_count(2.70, 0.77, right_top_only)
    put_count(2.70, -0.90, right_bottom_only)
    put_count(2.12, 0.00, right_top_right_bottom_only)
    put_count(1.12, 0.48, center_right_top_only)
    put_count(1.12, -0.48, center_right_bottom_only)

    if title is not None:
        fig.suptitle(title, fontsize=11, y=0.991, fontweight="normal")
    ax.set_xlim(-3.555, 3.555)
    ax.set_ylim(-2.235, 2.235)
    if save_path is not None:
        _render_article(fig, ACTIVE_CHAIN, "overlap")
    else:
        plt.show()

    return {
        "center_only_after_all": center_only_after_all,
        "left_triple": left_triple,
        "right_triple": right_triple,
        "left_regions": {
            "left_top_only": left_top_only,
            "left_bottom_only": left_bottom_only,
            "left_top_left_bottom_only": left_top_left_bottom_only,
            "center_left_top_only": center_left_top_only,
            "center_left_bottom_only": center_left_bottom_only,
            "left_triple": left_triple,
        },
        "right_regions": {
            "right_top_only": right_top_only,
            "right_bottom_only": right_bottom_only,
            "right_top_right_bottom_only": right_top_right_bottom_only,
            "center_right_top_only": center_right_top_only,
            "center_right_bottom_only": center_right_bottom_only,
            "right_triple": right_triple,
        },
    }


g1_tra_aaV = set(ct_g1_aaV_tra.index)
g2_tra_aaV = set(ct_g2_aaV_tra.index)
g3_tra_aaV = set(ct_g3_aaV_tra.index)
g4_tra_aaV = set(ct_g4_aaV_tra.index)
g5_tra_aaV = set(ct_g5_aaV_tra.index)
g6_tra_aaV = set(ct_g6_aaV_tra.index)

res_g1_tra = plot_double_triple_venn_like(
    center_set=g1_tra_aaV,
    left_top_set=g5_tra_aaV,
    left_bottom_set=g6_tra_aaV,
    right_top_set=g2_tra_aaV,
    right_bottom_set=g4_tra_aaV,
    center_label="g1",
    left_top_label="g5",
    left_bottom_label="g6",
    right_top_label="g2",
    right_bottom_label="g4",
    title=f"TRA g1: aaV clonotype overlap with g5–g6 and g2–g4{FIGURE_STRATUM_SUFFIX}",
    save_path=str(FIGURE_DIR / "TRA_g1_clonotype_overlap.png"),
)

In [ ]:
def clone_set_to_df(clone_set, overlap_type="aaV"):
    clone_set = sorted(clone_set)
    if overlap_type == "aaV":
        return pd.DataFrame(clone_set, columns=["cdr3aa", "v"])
    if overlap_type == "aaVJ":
        return pd.DataFrame(clone_set, columns=["cdr3aa", "v", "j"])
    raise ValueError("overlap_type must be 'aaV' or 'aaVJ'")


g1 = set(ct_g1_aaV_tra.index)
g2 = set(ct_g2_aaV_tra.index)
g3 = set(ct_g3_aaV_tra.index)
g4 = set(ct_g4_aaV_tra.index)
g5 = set(ct_g5_aaV_tra.index)
g6 = set(ct_g6_aaV_tra.index)

g1_remaining_aaV_tra = g1 - (g2 | g4 | g5 | g6)
clones_g1_remaining_aaV_tra_df = clone_set_to_df(
    g1_remaining_aaV_tra,
    overlap_type="aaV",
)
display(
    clones_g1_remaining_aaV_tra_df["v"]
    .value_counts()
    .rename_axis("v")
    .reset_index(name="n_remaining_aaV")
    .head(20)
)


In [ ]:
def clone_set_to_df(clone_set, overlap_type="aaV"):
    if overlap_type == "aaV":
        return pd.DataFrame(sorted(clone_set), columns=["cdr3aa", "v"])

    if overlap_type == "aaVJ":
        return pd.DataFrame(sorted(clone_set), columns=["cdr3aa", "v", "j"])

    raise ValueError("overlap_type must be 'aaV' or 'aaVJ'")


def make_v_summary_from_clone_set(clone_set, total_n, overlap_type="aaV"):
    df = clone_set_to_df(clone_set, overlap_type=overlap_type)

    if df.empty:
        return pd.DataFrame(columns=["v", "n_cdr3", "frequency", "frequency_pct"])

    out = (
        df["v"]
        .value_counts()
        .rename_axis("v")
        .reset_index(name="n_cdr3")
    )

    out["frequency"] = out["n_cdr3"] / total_n
    out["frequency_pct"] = 100 * out["frequency"]

    return out


def compare_full_vs_remaining_after_subtractions(
    target_ct,
    subtract_cts,
    target_label,
    overlap_type="aaV"
):
    target_set = set(target_ct.index)
    subtract_union = set().union(*[set(ct.index) for ct in subtract_cts])

    remaining_set = target_set - subtract_union

    full_v = make_v_summary_from_clone_set(
        target_set,
        total_n=len(target_set),
        overlap_type=overlap_type
    )

    remaining_v = make_v_summary_from_clone_set(
        remaining_set,
        total_n=len(target_set),
        overlap_type=overlap_type
    )

    full_v = full_v.rename(columns={
        "n_cdr3": f"cdr3_in_full_{target_label}",
        "frequency": f"frequency_in_full_{target_label}",
        "frequency_pct": f"frequency_in_full_{target_label}_pct"
    })

    remaining_v = remaining_v.rename(columns={
        "n_cdr3": f"cdr3_remaining_in_{target_label}",
        "frequency": f"frequency_remaining_in_{target_label}_relative_to_original",
        "frequency_pct": f"frequency_remaining_in_{target_label}_pct_relative_to_original"
    })

    cmp_df = full_v.merge(remaining_v, on="v", how="left")

    remaining_count_col = f"cdr3_remaining_in_{target_label}"
    full_count_col = f"cdr3_in_full_{target_label}"

    remaining_freq_col = f"frequency_remaining_in_{target_label}_relative_to_original"
    remaining_freq_pct_col = f"frequency_remaining_in_{target_label}_pct_relative_to_original"
    full_freq_pct_col = f"frequency_in_full_{target_label}_pct"

    cmp_df[remaining_count_col] = cmp_df[remaining_count_col].fillna(0).astype(int)
    cmp_df[remaining_freq_col] = cmp_df[remaining_freq_col].fillna(0.0)
    cmp_df[remaining_freq_pct_col] = cmp_df[remaining_freq_pct_col].fillna(0.0)

    cmp_df["lost_cdr3"] = cmp_df[full_count_col] - cmp_df[remaining_count_col]

    cmp_df["frequency_change_pct"] = (
        cmp_df[full_freq_pct_col] - cmp_df[remaining_freq_pct_col]
    )

    cmp_df["retained_cdr3_share"] = (
        cmp_df[remaining_count_col] / cmp_df[full_count_col]
    )

    cmp_df["retained_cdr3_share_pct"] = (
        100 * cmp_df["retained_cdr3_share"]
    )

    cmp_df["loss_share"] = cmp_df["lost_cdr3"] / cmp_df[full_count_col]
    cmp_df["loss_share_pct"] = 100 * cmp_df["loss_share"]

    cmp_df_changed = (
        cmp_df[cmp_df["lost_cdr3"] != 0]
        .sort_values(
            ["loss_share", "lost_cdr3", full_count_col],
            ascending=[False, False, False]
        )
        .reset_index(drop=True)
    )

    return cmp_df, cmp_df_changed, remaining_set


cmp_g1_tra_all, cmp_g1_tra_changed_ranked, g1_remaining_aaV_tra = (
    compare_full_vs_remaining_after_subtractions(
        target_ct=ct_g1_aaV_tra,
        subtract_cts=[
            ct_g2_aaV_tra,
            ct_g4_aaV_tra,
            ct_g5_aaV_tra,
            ct_g6_aaV_tra
        ],
        target_label="g1",
        overlap_type="aaV"
    )
)

display(cmp_g1_tra_changed_ranked.head(20))

cmp_g1_tra_changed_ranked.to_csv(
    RESULT_DIR / "TRA_cmp_g1_remaining_after_g2_g4_g5_g6_ranked.csv",
    index=False
)

cmp_g1_tra_all.to_csv(RESULT_DIR / "tra_v_retention.csv", index=False)


In [ ]:
def make_clone_intersection_table_venn3(
    ct_a,
    ct_b,
    ct_c,
    label_a="g1",
    label_b="g5",
    label_c="g6",
    overlap_type="aaV"
):
    set_a = set(ct_a.index)
    set_b = set(ct_b.index)
    set_c = set(ct_c.index)

    all_clones = set_a | set_b | set_c

    sum_a = ct_a.sum(axis=1).astype(float).to_dict()
    sum_b = ct_b.sum(axis=1).astype(float).to_dict()
    sum_c = ct_c.sum(axis=1).astype(float).to_dict()

    total_a = float(ct_a.sum(axis=1).sum())
    total_b = float(ct_b.sum(axis=1).sum())
    total_c = float(ct_c.sum(axis=1).sum())

    region_names = {
        "100": f"{label_a} only",
        "010": f"{label_b} only",
        "001": f"{label_c} only",
        "110": f"({label_a}∩{label_b})-{label_c}",
        "101": f"({label_a}∩{label_c})-{label_b}",
        "011": f"({label_b}∩{label_c})-{label_a}",
        "111": f"{label_a}∩{label_b}∩{label_c}",
    }

    rows = []

    for clone in sorted(all_clones):
        in_a = clone in set_a
        in_b = clone in set_b
        in_c = clone in set_c

        region_id = (
            ("1" if in_a else "0") +
            ("1" if in_b else "0") +
            ("1" if in_c else "0")
        )

        if overlap_type == "aaV":
            cdr3aa, v = clone
            row = {
                "cdr3aa": cdr3aa,
                "v": v,
                "clone_id": clone,
            }

        elif overlap_type == "aaVJ":
            cdr3aa, v, j = clone
            row = {
                "cdr3aa": cdr3aa,
                "v": v,
                "j": j,
                "clone_id": clone,
            }

        else:
            raise ValueError("overlap_type must be 'aaV' or 'aaVJ'")

        row_sum_a = float(sum_a.get(clone, 0.0))
        row_sum_b = float(sum_b.get(clone, 0.0))
        row_sum_c = float(sum_c.get(clone, 0.0))

        row.update({
            f"in_{label_a}": in_a,
            f"in_{label_b}": in_b,
            f"in_{label_c}": in_c,

            "region_id": region_id,
            "region_name": region_names[region_id],

            f"row_sum_{label_a}": row_sum_a,
            f"row_sum_{label_b}": row_sum_b,
            f"row_sum_{label_c}": row_sum_c,

            f"freq_vs_{label_a}": row_sum_a / total_a if total_a > 0 else np.nan,
            f"freq_vs_{label_b}": row_sum_b / total_b if total_b > 0 else np.nan,
            f"freq_vs_{label_c}": row_sum_c / total_c if total_c > 0 else np.nan,
        })

        row[f"freq_vs_{label_a}_pct"] = 100 * row[f"freq_vs_{label_a}"]
        row[f"freq_vs_{label_b}_pct"] = 100 * row[f"freq_vs_{label_b}"]
        row[f"freq_vs_{label_c}_pct"] = 100 * row[f"freq_vs_{label_c}"]

        rows.append(row)

    out = (
        pd.DataFrame(rows)
        .sort_values(
            ["region_id", f"row_sum_{label_a}", f"row_sum_{label_b}", f"row_sum_{label_c}", "v", "cdr3aa"],
            ascending=[True, False, False, False, True, True]
        )
        .reset_index(drop=True)
    )

    return out

g1_g5_g6_intersection_table = make_clone_intersection_table_venn3(
    ct_a=ct_g1_aaV_tra,
    ct_b=ct_g5_aaV_tra,
    ct_c=ct_g6_aaV_tra,
    label_a="g1",
    label_b="g5",
    label_c="g6",
    overlap_type="aaV"
)

display(g1_g5_g6_intersection_table.head(20))

In [ ]:
def make_v_region_summary(intersection_df, freq_col):
    target_region_order = ["100", "110", "101", "111"]
    target_regions = intersection_df[
        intersection_df["region_id"].astype(str).isin(target_region_order)
    ].copy()
    out = (
        target_regions
        .groupby(["v", "region_id", "region_name"], as_index=False)
        .agg(
            n_clones=("clone_id", "nunique"),
            burden=(freq_col, "sum"),
            median_clone_freq=(freq_col, "median"),
            mean_clone_freq=(freq_col, "mean"),
        )
    )
    total_burden = out.groupby("v")["burden"].transform("sum")
    out["region_share_within_v"] = np.where(
        total_burden.gt(0), out["burden"] / total_burden, 0.0
    )
    return out


def make_v_profile(cmp_df, intersection_df, target_label, freq_col):
    target_region_order = ["100", "110", "101", "111"]
    region_summary = make_v_region_summary(intersection_df, freq_col=freq_col)
    share_wide = (
        region_summary
        .pivot(index="v", columns="region_id", values="region_share_within_v")
        .reindex(columns=target_region_order, fill_value=0.0)
        .fillna(0.0)
    )

    probabilities = share_wide[target_region_order].replace(0, np.nan)
    entropy = -(
        probabilities * np.log2(probabilities)
    ).sum(axis=1).fillna(0.0) / np.log2(len(target_region_order))
    share_wide["region_entropy"] = entropy

    profile = cmp_df.merge(share_wide.reset_index(), on="v", how="left").fillna(0)
    profile["private_share"] = profile.get("100", 0)
    profile["triple_share"] = profile.get("111", 0)

    full_frequency_col = f"frequency_in_full_{target_label}"
    profile["score_s"] = (
        profile[full_frequency_col] * profile["retained_cdr3_share"]
    ).round(12)
    profile["retention_score"] = profile["score_s"]
    profile["loss_score"] = profile[full_frequency_col] * profile["loss_share"]
    profile = profile.sort_values(
        ["score_s", full_frequency_col, "v"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    return profile, region_summary


g1_profile, g1_v_region_summary = make_v_profile(
    cmp_df=cmp_g1_tra_all,                      
    intersection_df=g1_g5_g6_intersection_table,
    target_label="g1",
    freq_col="freq_vs_g1"
)


In [ ]:
def _stratum_title():
    return "combined CD4+ and CD8+" if STRATUM == "all_combined" else STRATUM_LABEL


def _plot_thresholds(chain, panel, tables, *, cohort_tables=None):
    if V_THRESHOLD_MODE == "fixed":
        return FIXED_THRESHOLDS
    cutoffs, records = fit_mouse_dropout_thresholds(tables, metadata, cohort_tables=cohort_tables)
    V_THRESHOLD_RECORDS.extend({"chain": chain, "panel": panel, **row} for row in records)
    return cutoffs


def _render_structural_figures(profile, chain):
    thresholds = _plot_thresholds(chain, STRATUM, chain_tables)
    _render_article(plot_dumbbell(profile, chain, FIGURE_STRATUM_SUFFIX, thresholds=thresholds), chain, "dumbbell")
    bubble, genes = plot_retention(profile, chain, _stratum_title(), thresholds=thresholds)
    _render_article(bubble, chain, "retention")
    _render_article(plot_localization(profile, genes, chain, FIGURE_STRATUM_SUFFIX, thresholds=thresholds), chain, "localization")
    pd.DataFrame({"v": genes}).to_csv(RESULT_DIR / f"{chain.lower()}_bubble_labels.csv", index=False)


    return genes


tra_bubble_genes = _render_structural_figures(g1_profile, "TRA")

In [ ]:
g1_only_barplot_df = (
    g1_profile.loc[g1_profile["v"].isin(tra_bubble_genes), ["v", "100"]]
    .rename(columns={"100": "g1_only_fraction"})
    .sort_values("g1_only_fraction", ascending=False)
    .reset_index(drop=True)
)
display(g1_only_barplot_df)


## TRB exact-set analysis


In [ ]:
ACTIVE_CHAIN = "TRB"
baseline_eligible_samples = None
if RUN_MODE == "spike-in" and SPIKE_CASE != "baseline":
    baseline_chain_tables = load_snapshot_tables(RESULT_ROOT / "baseline", "TRB", metadata)
    baseline_eligible_samples = {
        group: {
            str(sample_id)
            for sample_id in table.columns
            if int(table[sample_id].sum()) >= DOWNSAMPLE_DEPTH
        }
        for group, table in baseline_chain_tables.items()
    }
    raw_chain_tables = {group: table.copy() for group, table in baseline_chain_tables.items()}
else:
    raw_chain_tables = {
        group: intersections.count_table(
            globals()[f"clonosets_trb_{group}"],
            cl_filter=FUNCTIONAL_UMI_FILTER,
            overlap_type="aaV",
            mismatches=0,
        )
        for group in ("g1", "g2", "g3", "g4", "g5", "g6")
        if RUN_MODE != "v-inputs" or group != "g3"
    }
    if RUN_MODE == "spike-in":
        snapshot_tables(raw_chain_tables, RESULT_DIR, "TRB", metadata)

if SPIKE_SELECTION is not None and SPIKE_CHAIN == "TRB":
    raw_chain_tables, SPIKE_DOSE_TABLE = apply_spike_to_tables(
        raw_chain_tables, SPIKE_SELECTION, SPIKE_FRACTION
    )
    SPIKE_DOSE_TABLE.to_csv(RESULT_DIR / "spike_dose_by_sample.csv", index=False)

if RUN_MODE == "v-inputs" or V_THRESHOLD_MODE == "adaptive":
    raw_chain_tables = {group: table.sort_index() for group, table in raw_chain_tables.items()}

chain_tables, trb_depth_audit = downsample_group_tables(
    raw_chain_tables,
    depth=DOWNSAMPLE_DEPTH,
    seed=ACTIVE_STRUCTURAL_SEED,
    eligible_by_group=baseline_eligible_samples,
)
trb_depth_audit.to_csv(RESULT_DIR / "trb_structural_depth_audit.csv", index=False)

for group in raw_chain_tables:
    globals()[f"raw_ct_{group}_aaV_trb"] = raw_chain_tables[group]
    globals()[f"ct_{group}_aaV_trb"] = chain_tables[group]

if RUN_MODE == "v-inputs":
    metadata.to_csv(RESULT_DIR / "mouse_metadata.csv", index=False)
    snapshot_tables(raw_chain_tables, RESULT_DIR / "raw", ACTIVE_CHAIN, metadata)
    snapshot_tables(chain_tables, RESULT_DIR / "structural", ACTIVE_CHAIN, metadata)
    (RESULT_DIR / "v_input_seeds.json").write_text(json.dumps({
        "baseline_results": str(baseline_results),
        "structural_seed": ACTIVE_STRUCTURAL_SEED, "calibration_seed": DOWNSAMPLE_SEED,
        "depth": DOWNSAMPLE_DEPTH,
    }, indent=2) + "\n")

print(
    f"TRB structural sets use functional clonotypes downsampled to "
    f"{DOWNSAMPLE_DEPTH:,} UMI per eligible library with seed {ACTIVE_STRUCTURAL_SEED}"
)

### TRB overlap regions and retained V segments


In [ ]:
g1_trb_aaV = set(ct_g1_aaV_trb.index)
g2_trb_aaV = set(ct_g2_aaV_trb.index)
g3_trb_aaV = set(ct_g3_aaV_trb.index)
g4_trb_aaV = set(ct_g4_aaV_trb.index)
g5_trb_aaV = set(ct_g5_aaV_trb.index)
g6_trb_aaV = set(ct_g6_aaV_trb.index)

res_g1_trb = plot_double_triple_venn_like(
    center_set=g1_trb_aaV,
    left_top_set=g5_trb_aaV,
    left_bottom_set=g6_trb_aaV,
    right_top_set=g2_trb_aaV,
    right_bottom_set=g4_trb_aaV,
    center_label="g1",
    left_top_label="g5",
    left_bottom_label="g6",
    right_top_label="g2",
    right_bottom_label="g4",
    title=f"TRB g1: aaV clonotype overlap with g5–g6 and g2–g4{FIGURE_STRATUM_SUFFIX}",
    save_path=str(FIGURE_DIR / "TRB_g1_clonotype_overlap.png"),
)

In [ ]:
def clone_set_to_df(clone_set, overlap_type="aaV"):
    clone_set = sorted(clone_set)
    if overlap_type == "aaV":
        return pd.DataFrame(clone_set, columns=["cdr3aa", "v"])
    if overlap_type == "aaVJ":
        return pd.DataFrame(clone_set, columns=["cdr3aa", "v", "j"])
    raise ValueError("overlap_type must be 'aaV' or 'aaVJ'")


g1 = set(ct_g1_aaV_trb.index)
g2 = set(ct_g2_aaV_trb.index)
g3 = set(ct_g3_aaV_trb.index)
g4 = set(ct_g4_aaV_trb.index)
g5 = set(ct_g5_aaV_trb.index)
g6 = set(ct_g6_aaV_trb.index)

g1_remaining_aaV_trb = g1 - (g2 | g4 | g5 | g6)
clones_g1_remaining_aaV_trb_df = clone_set_to_df(
    g1_remaining_aaV_trb,
    overlap_type="aaV",
)
display(
    clones_g1_remaining_aaV_trb_df["v"]
    .value_counts()
    .rename_axis("v")
    .reset_index(name="n_remaining_aaV")
    .head(20)
)


### TRB V-segment retention tables


In [ ]:
def clone_set_to_df(clone_set, overlap_type="aaV"):
    if overlap_type == "aaV":
        return pd.DataFrame(sorted(clone_set), columns=["cdr3aa", "v"])

    if overlap_type == "aaVJ":
        return pd.DataFrame(sorted(clone_set), columns=["cdr3aa", "v", "j"])

    raise ValueError("overlap_type must be 'aaV' or 'aaVJ'")


def make_v_summary_from_clone_set(clone_set, total_n, overlap_type="aaV"):
    df = clone_set_to_df(clone_set, overlap_type=overlap_type)

    if df.empty:
        return pd.DataFrame(columns=["v", "n_cdr3", "frequency", "frequency_pct"])

    out = (
        df["v"]
        .value_counts()
        .rename_axis("v")
        .reset_index(name="n_cdr3")
    )

    out["frequency"] = out["n_cdr3"] / total_n
    out["frequency_pct"] = 100 * out["frequency"]

    return out


def compare_full_vs_remaining_after_subtractions(
    target_ct,
    subtract_cts,
    target_label,
    overlap_type="aaV"
):
    target_set = set(target_ct.index)
    subtract_union = set().union(*[set(ct.index) for ct in subtract_cts])

    remaining_set = target_set - subtract_union

    full_v = make_v_summary_from_clone_set(
        target_set,
        total_n=len(target_set),
        overlap_type=overlap_type
    )

    remaining_v = make_v_summary_from_clone_set(
        remaining_set,
        total_n=len(target_set),
        overlap_type=overlap_type
    )

    full_v = full_v.rename(columns={
        "n_cdr3": f"cdr3_in_full_{target_label}",
        "frequency": f"frequency_in_full_{target_label}",
        "frequency_pct": f"frequency_in_full_{target_label}_pct"
    })

    remaining_v = remaining_v.rename(columns={
        "n_cdr3": f"cdr3_remaining_in_{target_label}",
        "frequency": f"frequency_remaining_in_{target_label}_relative_to_original",
        "frequency_pct": f"frequency_remaining_in_{target_label}_pct_relative_to_original"
    })

    cmp_df = full_v.merge(remaining_v, on="v", how="left")

    remaining_count_col = f"cdr3_remaining_in_{target_label}"
    full_count_col = f"cdr3_in_full_{target_label}"

    remaining_freq_col = f"frequency_remaining_in_{target_label}_relative_to_original"
    remaining_freq_pct_col = f"frequency_remaining_in_{target_label}_pct_relative_to_original"
    full_freq_pct_col = f"frequency_in_full_{target_label}_pct"

    cmp_df[remaining_count_col] = cmp_df[remaining_count_col].fillna(0).astype(int)
    cmp_df[remaining_freq_col] = cmp_df[remaining_freq_col].fillna(0.0)
    cmp_df[remaining_freq_pct_col] = cmp_df[remaining_freq_pct_col].fillna(0.0)

    cmp_df["lost_cdr3"] = cmp_df[full_count_col] - cmp_df[remaining_count_col]

    cmp_df["frequency_change_pct"] = (
        cmp_df[full_freq_pct_col] - cmp_df[remaining_freq_pct_col]
    )

    cmp_df["retained_cdr3_share"] = (
        cmp_df[remaining_count_col] / cmp_df[full_count_col]
    )

    cmp_df["retained_cdr3_share_pct"] = (
        100 * cmp_df["retained_cdr3_share"]
    )

    cmp_df["loss_share"] = cmp_df["lost_cdr3"] / cmp_df[full_count_col]
    cmp_df["loss_share_pct"] = 100 * cmp_df["loss_share"]

    cmp_df_changed = (
        cmp_df[cmp_df["lost_cdr3"] != 0]
        .sort_values(
            ["loss_share", "lost_cdr3", full_count_col],
            ascending=[False, False, False]
        )
        .reset_index(drop=True)
    )

    return cmp_df, cmp_df_changed, remaining_set

cmp_g1_all, cmp_g1_changed_ranked, g1_remaining_aaV_trb = (
    compare_full_vs_remaining_after_subtractions(
        target_ct=ct_g1_aaV_trb,
        subtract_cts=[
            ct_g2_aaV_trb,
            ct_g4_aaV_trb,
            ct_g5_aaV_trb,
            ct_g6_aaV_trb
        ],
        target_label="g1",
        overlap_type="aaV"
    )
)

display(cmp_g1_changed_ranked.head(20))

cmp_g1_changed_ranked.to_csv(
    RESULT_DIR / "TRB_cmp_g1_remaining_after_g2_g4_g5_g6_ranked.csv",
    index=False
)


cmp_g1_all.to_csv(RESULT_DIR / "trb_v_retention.csv", index=False)

The following table supports the V-segment retention figures.


In [ ]:
def make_clone_intersection_table_venn3(
    ct_a,
    ct_b,
    ct_c,
    label_a="g1",
    label_b="g5",
    label_c="g6",
    overlap_type="aaV"
):
    set_a = set(ct_a.index)
    set_b = set(ct_b.index)
    set_c = set(ct_c.index)

    all_clones = set_a | set_b | set_c

    sum_a = ct_a.sum(axis=1).astype(float).to_dict()
    sum_b = ct_b.sum(axis=1).astype(float).to_dict()
    sum_c = ct_c.sum(axis=1).astype(float).to_dict()

    total_a = float(ct_a.sum(axis=1).sum())
    total_b = float(ct_b.sum(axis=1).sum())
    total_c = float(ct_c.sum(axis=1).sum())

    region_names = {
        "100": f"{label_a} only",
        "010": f"{label_b} only",
        "001": f"{label_c} only",
        "110": f"({label_a}∩{label_b})-{label_c}",
        "101": f"({label_a}∩{label_c})-{label_b}",
        "011": f"({label_b}∩{label_c})-{label_a}",
        "111": f"{label_a}∩{label_b}∩{label_c}",
    }

    rows = []

    for clone in sorted(all_clones):
        in_a = clone in set_a
        in_b = clone in set_b
        in_c = clone in set_c

        region_id = (
            ("1" if in_a else "0") +
            ("1" if in_b else "0") +
            ("1" if in_c else "0")
        )

        if overlap_type == "aaV":
            cdr3aa, v = clone
            row = {
                "cdr3aa": cdr3aa,
                "v": v,
                "clone_id": clone,
            }

        elif overlap_type == "aaVJ":
            cdr3aa, v, j = clone
            row = {
                "cdr3aa": cdr3aa,
                "v": v,
                "j": j,
                "clone_id": clone,
            }

        else:
            raise ValueError("overlap_type must be 'aaV' or 'aaVJ'")

        row_sum_a = float(sum_a.get(clone, 0.0))
        row_sum_b = float(sum_b.get(clone, 0.0))
        row_sum_c = float(sum_c.get(clone, 0.0))

        row.update({
            f"in_{label_a}": in_a,
            f"in_{label_b}": in_b,
            f"in_{label_c}": in_c,

            "region_id": region_id,
            "region_name": region_names[region_id],

            f"row_sum_{label_a}": row_sum_a,
            f"row_sum_{label_b}": row_sum_b,
            f"row_sum_{label_c}": row_sum_c,

            f"freq_vs_{label_a}": row_sum_a / total_a if total_a > 0 else np.nan,
            f"freq_vs_{label_b}": row_sum_b / total_b if total_b > 0 else np.nan,
            f"freq_vs_{label_c}": row_sum_c / total_c if total_c > 0 else np.nan,
        })

        row[f"freq_vs_{label_a}_pct"] = 100 * row[f"freq_vs_{label_a}"]
        row[f"freq_vs_{label_b}_pct"] = 100 * row[f"freq_vs_{label_b}"]
        row[f"freq_vs_{label_c}_pct"] = 100 * row[f"freq_vs_{label_c}"]

        rows.append(row)

    out = (
        pd.DataFrame(rows)
        .sort_values(
            ["region_id", f"row_sum_{label_a}", f"row_sum_{label_b}", f"row_sum_{label_c}", "v", "cdr3aa"],
            ascending=[True, False, False, False, True, True]
        )
        .reset_index(drop=True)
    )

    return out

g1_g5_g6_intersection_table = make_clone_intersection_table_venn3(
    ct_a=ct_g1_aaV_trb,
    ct_b=ct_g5_aaV_trb,
    ct_c=ct_g6_aaV_trb,
    label_a="g1",
    label_b="g5",
    label_c="g6",
    overlap_type="aaV"
)

display(g1_g5_g6_intersection_table.head(20))

### TRB V-segment retention figures


In [ ]:
def make_v_region_summary(intersection_df, freq_col):
    target_region_order = ["100", "110", "101", "111"]
    target_regions = intersection_df[
        intersection_df["region_id"].astype(str).isin(target_region_order)
    ].copy()
    out = (
        target_regions
        .groupby(["v", "region_id", "region_name"], as_index=False)
        .agg(
            n_clones=("clone_id", "nunique"),
            burden=(freq_col, "sum"),
            median_clone_freq=(freq_col, "median"),
            mean_clone_freq=(freq_col, "mean"),
        )
    )
    total_burden = out.groupby("v")["burden"].transform("sum")
    out["region_share_within_v"] = np.where(
        total_burden.gt(0), out["burden"] / total_burden, 0.0
    )
    return out


def make_v_profile(cmp_df, intersection_df, target_label, freq_col):
    target_region_order = ["100", "110", "101", "111"]
    region_summary = make_v_region_summary(intersection_df, freq_col=freq_col)
    share_wide = (
        region_summary
        .pivot(index="v", columns="region_id", values="region_share_within_v")
        .reindex(columns=target_region_order, fill_value=0.0)
        .fillna(0.0)
    )

    probabilities = share_wide[target_region_order].replace(0, np.nan)
    entropy = -(
        probabilities * np.log2(probabilities)
    ).sum(axis=1).fillna(0.0) / np.log2(len(target_region_order))
    share_wide["region_entropy"] = entropy

    profile = cmp_df.merge(share_wide.reset_index(), on="v", how="left").fillna(0)
    profile["private_share"] = profile.get("100", 0)
    profile["triple_share"] = profile.get("111", 0)

    full_frequency_col = f"frequency_in_full_{target_label}"
    profile["score_s"] = (
        profile[full_frequency_col] * profile["retained_cdr3_share"]
    ).round(12)
    profile["retention_score"] = profile["score_s"]
    profile["loss_score"] = profile[full_frequency_col] * profile["loss_share"]
    profile = profile.sort_values(
        ["score_s", full_frequency_col, "v"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    return profile, region_summary


g1_profile, g1_v_region_summary = make_v_profile(
    cmp_df=cmp_g1_all,                      
    intersection_df=g1_g5_g6_intersection_table,
    target_label="g1",
    freq_col="freq_vs_g1"
)


In [ ]:
_render_structural_figures(g1_profile, "TRB")

## TRA/TRB comparison using Jensen-Shannon similarity and rank correlation


In [ ]:
ACTIVE_CHAIN = "TRA and TRB"
from itertools import combinations
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import jensenshannon


out_dir = str(RESULT_DIR / "tra_trb_pairwise_comparison")
os.makedirs(out_dir, exist_ok=True)

ct_aaV_by_group_chain = {
    "g1": {
        "TRA": ct_g1_aaV_tra,
        "TRB": ct_g1_aaV_trb,
    },
    "g2": {
        "TRA": ct_g2_aaV_tra,
        "TRB": ct_g2_aaV_trb,
    },
    "g5": {
        "TRA": ct_g5_aaV_tra,
        "TRB": ct_g5_aaV_trb,
    },
}

def infer_mouse_id_from_sample_id(sample_id):
    
    sample_id = str(sample_id)
    m = re.search(r"(g\d+)_m(\d+)", sample_id)

    if m is None:
        return sample_id

    return f"{m.group(1)}_m{m.group(2)}"


def prepare_metadata_mouse_id(metadata):
    md = metadata.copy()

    if "mouse_id" not in md.columns:
        md["mouse_id"] = md["sample_id"].apply(infer_mouse_id_from_sample_id)

    md["sample_id"] = md["sample_id"].astype(str)
    md["mouse_id"] = md["mouse_id"].astype(str)

    return md


def index_value_to_tuple(x):
    
    if isinstance(x, tuple):
        return x

    if isinstance(x, str):
        try:
            y = ast.literal_eval(x)
            if isinstance(y, tuple):
                return y
        except Exception:
            pass

    return x


def get_v_from_index(ct):
    
    if isinstance(ct.index, pd.MultiIndex):
        return ct.index.get_level_values(1).astype(str).tolist()

    out = []
    for x in ct.index:
        x = index_value_to_tuple(x)

        if not isinstance(x, tuple) or len(x) < 2:
            raise ValueError(
                "Cannot extract the V segment. "
                "Expected an aaV index of (cdr3aa, v)"
            )

        out.append(str(x[1]))

    return out


def make_feature_by_sample_matrix(ct, feature_mode="v"):
    
    x = ct.copy()
    x = x.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    if feature_mode == "aaV":
        x.index = [index_value_to_tuple(i) for i in x.index]
        return x

    if feature_mode == "v":
        v_genes = get_v_from_index(x)
        x["feature_id"] = v_genes

        out = (
            x
            .groupby("feature_id")
            .sum()
        )

        return out

    raise ValueError("feature_mode must be 'v' or 'aaV'")
    
def aggregate_samples_to_mouse(feature_by_sample, metadata):
    
    md = prepare_metadata_mouse_id(metadata)

    sample_to_mouse = dict(zip(md["sample_id"], md["mouse_id"]))

    mat = feature_by_sample.copy()
    mat.columns = mat.columns.astype(str)

    
    for sample_id in mat.columns:
        if sample_id not in sample_to_mouse:
            sample_to_mouse[sample_id] = infer_mouse_id_from_sample_id(sample_id)

    mouse_cols = [sample_to_mouse[c] for c in mat.columns]

    mat_by_mouse = mat.copy()
    mat_by_mouse.columns = mouse_cols

    
    mat_by_mouse = mat_by_mouse.T.groupby(level=0).sum()

    return mat_by_mouse


def normalize_rows_to_freq(mat):
    
    mat = mat.copy().astype(float)
    row_sums = mat.sum(axis=1)

    freq = mat.div(row_sums.replace(0, np.nan), axis=0).fillna(0.0)

    return freq

def pair_similarity(x, y, metric="js"):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if metric == "js":
        
        
        
        
        if x.sum() == 0 or y.sum() == 0:
            return np.nan

        x = x / x.sum()
        y = y / y.sum()

        return 1.0 - jensenshannon(x, y, base=2)

    if metric == "pearson":
        if np.std(x) == 0 or np.std(y) == 0:
            return np.nan
        return pearsonr(x, y)[0]

    if metric == "spearman":
        if np.std(x) == 0 or np.std(y) == 0:
            return np.nan
        return spearmanr(x, y)[0]

    if metric == "jaccard":
        xb = x > 0
        yb = y > 0

        union = np.logical_or(xb, yb).sum()
        inter = np.logical_and(xb, yb).sum()

        if union == 0:
            return np.nan

        return inter / union

    raise ValueError("metric must be 'js', 'pearson', 'spearman' or 'jaccard'")


def compute_similarity_matrix(mouse_feature_mat, metric="js"):
    ids = list(mouse_feature_mat.index)
    sim = pd.DataFrame(np.nan, index=ids, columns=ids)

    for a in ids:
        for b in ids:
            if a == b:
                sim.loc[a, b] = 1.0
            elif pd.isna(sim.loc[a, b]):
                value = pair_similarity(
                    mouse_feature_mat.loc[a].values,
                    mouse_feature_mat.loc[b].values,
                    metric=metric
                )
                sim.loc[a, b] = value
                sim.loc[b, a] = value

    return sim


def similarity_matrix_to_pairs(sim_mat, chain_label):
    rows = []

    for a, b in combinations(sim_mat.index, 2):
        rows.append({
            "mouse_a": a,
            "mouse_b": b,
            "pair_id": tuple(sorted((a, b))),
            f"similarity_{chain_label}": sim_mat.loc[a, b]
        })

    return pd.DataFrame(rows)

def compare_tra_trb_from_count_tables(
    ct_tra,
    ct_trb,
    metadata,
    group_no,
    feature_mode="v",
    metric="js"
):
    

    tra_feature_sample = make_feature_by_sample_matrix(
        ct_tra,
        feature_mode=feature_mode
    )

    trb_feature_sample = make_feature_by_sample_matrix(
        ct_trb,
        feature_mode=feature_mode
    )

    tra_mouse = aggregate_samples_to_mouse(
        tra_feature_sample,
        metadata=metadata
    )

    trb_mouse = aggregate_samples_to_mouse(
        trb_feature_sample,
        metadata=metadata
    )

    common_mice = sorted(set(tra_mouse.index) & set(trb_mouse.index))

    if len(common_mice) < 2:
        raise ValueError(
            f"{group_no}: fewer than two mice are shared between TRA and TRB. "
            f"TRA mice: {list(tra_mouse.index)}. "
            f"TRB mice: {list(trb_mouse.index)}."
        )

    tra_mouse = tra_mouse.loc[common_mice]
    trb_mouse = trb_mouse.loc[common_mice]

    if metric in ["js", "pearson", "spearman"]:
        tra_for_similarity = normalize_rows_to_freq(tra_mouse)
        trb_for_similarity = normalize_rows_to_freq(trb_mouse)
    else:
        tra_for_similarity = tra_mouse.copy()
        trb_for_similarity = trb_mouse.copy()

    sim_tra = compute_similarity_matrix(
        tra_for_similarity,
        metric=metric
    )

    sim_trb = compute_similarity_matrix(
        trb_for_similarity,
        metric=metric
    )

    pairs_tra = similarity_matrix_to_pairs(sim_tra, "TRA")
    pairs_trb = similarity_matrix_to_pairs(sim_trb, "TRB")

    pair_df = pairs_tra.merge(
        pairs_trb[["pair_id", "similarity_TRB"]],
        on="pair_id",
        how="inner"
    )

    ok = pair_df[["similarity_TRA", "similarity_TRB"]].dropna()

    if len(ok) >= 2:
        pearson_r, pearson_p = pearsonr(ok["similarity_TRA"], ok["similarity_TRB"])
        spearman_rho, spearman_p = spearmanr(ok["similarity_TRA"], ok["similarity_TRB"])
    else:
        pearson_r, pearson_p = np.nan, np.nan
        spearman_rho, spearman_p = np.nan, np.nan

    summary = pd.DataFrame([{
        "group_no": group_no,
        "feature_mode": feature_mode,
        "metric": metric,
        "n_common_mice": len(common_mice),
        "n_pairs": len(ok),
        "mean_similarity_TRA": ok["similarity_TRA"].mean(),
        "mean_similarity_TRB": ok["similarity_TRB"].mean(),
        "median_similarity_TRA": ok["similarity_TRA"].median(),
        "median_similarity_TRB": ok["similarity_TRB"].median(),
        "TRA_vs_TRB_pearson_r": pearson_r,
        "TRA_vs_TRB_pearson_p": pearson_p,
        "TRA_vs_TRB_spearman_rho": spearman_rho,
        "TRA_vs_TRB_spearman_p": spearman_p,
    }])

    return {
        "group_no": group_no,
        "feature_mode": feature_mode,
        "metric": metric,
        "common_mice": common_mice,
        "matrix_TRA": tra_for_similarity,
        "matrix_TRB": trb_for_similarity,
        "sim_TRA": sim_tra,
        "sim_TRB": sim_trb,
        "pair_df": pair_df,
        "summary": summary,
    }


    
    
def collect_pairwise_similarity_boxplot_table(results_dict):
    rows = []

    for group_no, result in results_dict.items():
        pair_df = result["pair_df"].copy()

        for value in pair_df["similarity_TRA"].dropna():
            rows.append({
                "group_no": group_no,
                "chain": "TRA",
                "similarity": value,
                "feature_mode": result["feature_mode"],
                "metric": result["metric"]
            })

        for value in pair_df["similarity_TRB"].dropna():
            rows.append({
                "group_no": group_no,
                "chain": "TRB",
                "similarity": value,
                "feature_mode": result["feature_mode"],
                "metric": result["metric"]
            })

    return pd.DataFrame(rows)


In [ ]:
ACTIVE_CHAIN = "TRA and TRB"
groups = ["g1", "g5"]
results_v_js = {}
similarity_status = []

for group_no in groups:
    try:
        result = compare_tra_trb_from_count_tables(
            ct_tra=globals()[f"ct_{group_no}_aaV_tra"],
            ct_trb=globals()[f"ct_{group_no}_aaV_trb"],
            metadata=metadata,
            group_no=group_no,
            feature_mode="v",
            metric="js",
        )
    except ValueError as error:
        similarity_status.append(
            {"group_no": group_no, "status": "not estimated", "reason": str(error)}
        )
        print(f"Skipping {group_no} TRA/TRB similarity: {error}")
        continue

    results_v_js[group_no] = result
    similarity_status.append(
        {"group_no": group_no, "status": "estimated", "reason": ""}
    )
    display(result["summary"])
    result["summary"].to_csv(
        f"{out_dir}/{group_no}_TRA_TRB_V_usage_JS_summary.csv", index=False
    )
    result["pair_df"].to_csv(
        f"{out_dir}/{group_no}_TRA_TRB_V_usage_JS_pairwise.csv", index=False
    )

pd.DataFrame(similarity_status).to_csv(
    f"{out_dir}/TRA_TRB_V_usage_JS_status.csv", index=False
)
if results_v_js:
    box_v_js = collect_pairwise_similarity_boxplot_table(results_v_js)
    box_v_js.to_csv(
        f"{out_dir}/TRA_TRB_V_usage_JS_pairwise_table.csv", index=False
    )

print("TRA/TRB similarity tables saved; auxiliary pairwise figures are omitted.")


## Mouse-level TRA/TRB comparison using UMI counts


In [ ]:
ACTIVE_CHAIN = "TRA and TRB"
import os
import re
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import combinations
from scipy.stats import pearsonr, spearmanr







OUT_DIR = str(RESULT_DIR / "tra_trb_pairwise_umi_clone_correlations")
os.makedirs(OUT_DIR, exist_ok=True)

GROUPS_TO_RUN = ["g1", "g5", "g6"]






TRANSFORM = "log1p"


MIN_CLONES_UNION = 10

SHOW_FIGURES = True






def get_existing_var(var_name):
    return globals().get(var_name, None)


def build_ct_pair_plan(groups_to_run):
    
    plan = {}

    for group_no in groups_to_run:
        ct_tra_name = f"ct_{group_no}_aaV_tra"
        ct_trb_name = f"ct_{group_no}_aaV_trb"

        ct_tra = get_existing_var(ct_tra_name)
        ct_trb = get_existing_var(ct_trb_name)

        if ct_tra is None:
            print(f"Skipping {group_no}: missing {ct_tra_name}")
            continue

        if ct_trb is None:
            print(f"Skipping {group_no}: missing {ct_trb_name}")
            continue

        plan[group_no] = {
            "TRA": ct_tra,
            "TRB": ct_trb,
            "ct_tra_name": ct_tra_name,
            "ct_trb_name": ct_trb_name,
        }

    return plan






def infer_mouse_id_from_sample_id(sample_id):
    
    sample_id = str(sample_id)
    m = re.search(r"(g\d+)_m(\d+)", sample_id)

    if m is None:
        return sample_id

    return f"{m.group(1)}_m{m.group(2)}"


def prepare_metadata_mouse_id(metadata):
    md = metadata.copy()

    if "sample_id" not in md.columns:
        raise ValueError("Metadata does not contain a sample_id column")

    md["sample_id"] = md["sample_id"].astype(str)

    if "mouse_id" not in md.columns:
        md["mouse_id"] = md["sample_id"].apply(infer_mouse_id_from_sample_id)
    else:
        md["mouse_id"] = md["mouse_id"].astype(str)

    return md


def parse_index_value(x):
    
    if isinstance(x, tuple):
        return x

    if isinstance(x, str):
        try:
            y = ast.literal_eval(x)
            if isinstance(y, tuple):
                return y
        except Exception:
            pass

    return x


def get_clone_ids_from_count_table(ct):
    
    if isinstance(ct.index, pd.MultiIndex):
        clone_ids = list(ct.index.to_flat_index())
    else:
        clone_ids = [parse_index_value(x) for x in ct.index]

    return clone_ids


def count_table_to_mouse_clone_count_matrix(ct, metadata, group_no=None):
    
    md = prepare_metadata_mouse_id(metadata)

    if group_no is not None and "group_no" in md.columns:
        md = md[md["group_no"] == group_no].copy()

    sample_to_mouse = dict(zip(md["sample_id"], md["mouse_id"]))

    x = ct.copy()
    x.columns = x.columns.astype(str)

    sample_cols = [c for c in x.columns if c in sample_to_mouse]

    
    
    if len(sample_cols) == 0:
        sample_cols = list(x.columns)
        sample_to_mouse = {
            sample_id: infer_mouse_id_from_sample_id(sample_id)
            for sample_id in sample_cols
        }

    x = x[sample_cols].copy()
    x = x.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    clone_ids = get_clone_ids_from_count_table(x)
    x.index = pd.Index(clone_ids, dtype="object")

    x.columns = [sample_to_mouse[c] for c in x.columns]

    
    mouse_clone_mat = x.T.groupby(level=0).sum().T

    return mouse_clone_mat






def transform_pair_vectors(x, y, transform="log1p"):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if transform == "raw":
        return x, y

    if transform == "log1p":
        return np.log1p(x), np.log1p(y)

    if transform == "freq":
        sx = x.sum()
        sy = y.sum()

        x = x / sx if sx > 0 else x
        y = y / sy if sy > 0 else y

        return x, y

    if transform == "log1p_freq":
        sx = x.sum()
        sy = y.sum()

        x = x / sx if sx > 0 else x
        y = y / sy if sy > 0 else y

        return np.log1p(x), np.log1p(y)

    raise ValueError("transform must be one of: raw, log1p, freq, log1p_freq")


def safe_corr(x, y, method="pearson"):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) < 2:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    if method == "pearson":
        r, p = pearsonr(x, y)
        return r, p

    if method == "spearman":
        rho, p = spearmanr(x, y)
        return rho, p

    raise ValueError("method must be pearson or spearman")


def compute_pairwise_clone_umi_correlations(
    mouse_clone_mat,
    chain_label,
    group_no,
    transform="log1p",
    min_clones_union=10
):
    
    mouse_ids = sorted(mouse_clone_mat.columns.astype(str))
    mat = mouse_clone_mat.copy()
    mat.columns = mat.columns.astype(str)

    rows = []

    for mouse_a, mouse_b in combinations(mouse_ids, 2):
        raw_a = mat[mouse_a].astype(float).values
        raw_b = mat[mouse_b].astype(float).values

        
        
        union_mask = (raw_a > 0) | (raw_b > 0)
        shared_mask = (raw_a > 0) & (raw_b > 0)

        n_union = int(union_mask.sum())
        n_shared = int(shared_mask.sum())

        if n_union < min_clones_union:
            pearson_r, pearson_p = np.nan, np.nan
            spearman_rho, spearman_p = np.nan, np.nan
        else:
            x = raw_a[union_mask]
            y = raw_b[union_mask]

            x_t, y_t = transform_pair_vectors(
                x,
                y,
                transform=transform
            )

            pearson_r, pearson_p = safe_corr(
                x_t,
                y_t,
                method="pearson"
            )

            spearman_rho, spearman_p = safe_corr(
                x_t,
                y_t,
                method="spearman"
            )

        rows.append({
            "group_no": group_no,
            "chain": chain_label,
            "mouse_a": mouse_a,
            "mouse_b": mouse_b,
            "pair_id": tuple(sorted((mouse_a, mouse_b))),
            "transform": transform,
            "n_clones_union": n_union,
            "n_clones_shared": n_shared,
            "pct_shared_among_union": 100 * n_shared / n_union if n_union > 0 else np.nan,
            "pearson_umi_corr": pearson_r,
            "pearson_umi_p": pearson_p,
            "spearman_umi_rho": spearman_rho,
            "spearman_umi_p": spearman_p,
        })

    return pd.DataFrame(rows)


def pair_df_to_similarity_matrix(pair_df, value_col):
    mice = sorted(set(pair_df["mouse_a"]) | set(pair_df["mouse_b"]))

    sim = pd.DataFrame(
        np.eye(len(mice)),
        index=mice,
        columns=mice,
        dtype=float
    )

    for _, row in pair_df.iterrows():
        a = row["mouse_a"]
        b = row["mouse_b"]
        value = row[value_col]

        sim.loc[a, b] = value
        sim.loc[b, a] = value

    return sim


def compare_tra_trb_umi_correlations_for_group(
    ct_tra,
    ct_trb,
    metadata,
    group_no,
    transform="log1p",
    min_clones_union=10
):
    tra_mat = count_table_to_mouse_clone_count_matrix(
        ct=ct_tra,
        metadata=metadata,
        group_no=group_no
    )

    trb_mat = count_table_to_mouse_clone_count_matrix(
        ct=ct_trb,
        metadata=metadata,
        group_no=group_no
    )

    common_mice = sorted(
        set(tra_mat.columns.astype(str)) &
        set(trb_mat.columns.astype(str))
    )

    if len(common_mice) < 2:
        raise ValueError(
            f"{group_no}: fewer than two mice are shared between TRA and TRB. "
            f"TRA mice: {list(tra_mat.columns)}. "
            f"TRB mice: {list(trb_mat.columns)}."
        )

    tra_mat = tra_mat[common_mice]
    trb_mat = trb_mat[common_mice]

    tra_pairs = compute_pairwise_clone_umi_correlations(
        mouse_clone_mat=tra_mat,
        chain_label="TRA",
        group_no=group_no,
        transform=transform,
        min_clones_union=min_clones_union
    )

    trb_pairs = compute_pairwise_clone_umi_correlations(
        mouse_clone_mat=trb_mat,
        chain_label="TRB",
        group_no=group_no,
        transform=transform,
        min_clones_union=min_clones_union
    )

    pair_df = tra_pairs.merge(
        trb_pairs[
            [
                "pair_id",
                "n_clones_union",
                "n_clones_shared",
                "pct_shared_among_union",
                "pearson_umi_corr",
                "pearson_umi_p",
                "spearman_umi_rho",
                "spearman_umi_p"
            ]
        ],
        on="pair_id",
        how="inner",
        suffixes=("_TRA", "_TRB")
    )

    
    
    

    ok_pearson = pair_df[
        ["pearson_umi_corr_TRA", "pearson_umi_corr_TRB"]
    ].dropna()

    if len(ok_pearson) >= 2:
        tra_trb_pearson_r_on_paircorr, tra_trb_pearson_p_on_paircorr = pearsonr(
            ok_pearson["pearson_umi_corr_TRA"],
            ok_pearson["pearson_umi_corr_TRB"]
        )

        tra_trb_spearman_rho_on_paircorr, tra_trb_spearman_p_on_paircorr = spearmanr(
            ok_pearson["pearson_umi_corr_TRA"],
            ok_pearson["pearson_umi_corr_TRB"]
        )
    else:
        tra_trb_pearson_r_on_paircorr = np.nan
        tra_trb_pearson_p_on_paircorr = np.nan
        tra_trb_spearman_rho_on_paircorr = np.nan
        tra_trb_spearman_p_on_paircorr = np.nan

    summary = pd.DataFrame([{
        "group_no": group_no,
        "transform": transform,
        "n_common_mice": len(common_mice),
        "n_pairs": len(pair_df),

        "mean_pearson_umi_corr_TRA": pair_df["pearson_umi_corr_TRA"].mean(),
        "mean_pearson_umi_corr_TRB": pair_df["pearson_umi_corr_TRB"].mean(),

        "median_pearson_umi_corr_TRA": pair_df["pearson_umi_corr_TRA"].median(),
        "median_pearson_umi_corr_TRB": pair_df["pearson_umi_corr_TRB"].median(),

        "mean_spearman_umi_rho_TRA": pair_df["spearman_umi_rho_TRA"].mean(),
        "mean_spearman_umi_rho_TRB": pair_df["spearman_umi_rho_TRB"].mean(),

        "median_spearman_umi_rho_TRA": pair_df["spearman_umi_rho_TRA"].median(),
        "median_spearman_umi_rho_TRB": pair_df["spearman_umi_rho_TRB"].median(),

        "TRA_vs_TRB_pearson_r_of_pairwise_pearson_corr": tra_trb_pearson_r_on_paircorr,
        "TRA_vs_TRB_pearson_p_of_pairwise_pearson_corr": tra_trb_pearson_p_on_paircorr,

        "TRA_vs_TRB_spearman_rho_of_pairwise_pearson_corr": tra_trb_spearman_rho_on_paircorr,
        "TRA_vs_TRB_spearman_p_of_pairwise_pearson_corr": tra_trb_spearman_p_on_paircorr,
    }])

    sim_tra_pearson = pair_df_to_similarity_matrix(
        pair_df,
        "pearson_umi_corr_TRA"
    )

    sim_trb_pearson = pair_df_to_similarity_matrix(
        pair_df,
        "pearson_umi_corr_TRB"
    )

    sim_tra_spearman = pair_df_to_similarity_matrix(
        pair_df,
        "spearman_umi_rho_TRA"
    )

    sim_trb_spearman = pair_df_to_similarity_matrix(
        pair_df,
        "spearman_umi_rho_TRB"
    )

    return {
        "group_no": group_no,
        "transform": transform,
        "common_mice": common_mice,

        "TRA_mouse_clone_mat": tra_mat,
        "TRB_mouse_clone_mat": trb_mat,

        "pair_df": pair_df,
        "summary": summary,

        "sim_TRA_pearson": sim_tra_pearson,
        "sim_TRB_pearson": sim_trb_pearson,

        "sim_TRA_spearman": sim_tra_spearman,
        "sim_TRB_spearman": sim_trb_spearman,
    }











def collect_boxplot_table_from_corr_results(results_dict, corr_type="pearson"):
    rows = []

    if corr_type == "pearson":
        tra_col = "pearson_umi_corr_TRA"
        trb_col = "pearson_umi_corr_TRB"
        metric_label = "Pearson UMI-count correlation"
    elif corr_type == "spearman":
        tra_col = "spearman_umi_rho_TRA"
        trb_col = "spearman_umi_rho_TRB"
        metric_label = "Spearman UMI-count correlation"
    else:
        raise ValueError("corr_type must be 'pearson' or 'spearman'")

    for group_no, result in results_dict.items():
        pair_df = result["pair_df"].copy()

        for value in pair_df[tra_col].dropna():
            rows.append({
                "group_no": group_no,
                "chain": "TRA",
                "correlation": value,
                "metric": metric_label,
                "transform": result["transform"]
            })

        for value in pair_df[trb_col].dropna():
            rows.append({
                "group_no": group_no,
                "chain": "TRB",
                "correlation": value,
                "metric": metric_label,
                "transform": result["transform"]
            })

    return pd.DataFrame(rows)








ct_plan = build_ct_pair_plan(GROUPS_TO_RUN)
umi_corr_results = {}
umi_status = []

for group_no, cts in ct_plan.items():
    print("\n" + "=" * 90)
    print(f"Analyzing {group_no}: TRA vs TRB pairwise UMI clonotype correlations")
    try:
        result = compare_tra_trb_umi_correlations_for_group(
            ct_tra=cts["TRA"],
            ct_trb=cts["TRB"],
            metadata=metadata,
            group_no=group_no,
            transform=TRANSFORM,
            min_clones_union=MIN_CLONES_UNION,
        )
    except ValueError as error:
        umi_status.append(
            {"group_no": group_no, "status": "not estimated", "reason": str(error)}
        )
        print(f"Skipping {group_no} UMI correlation: {error}")
        continue

    umi_corr_results[group_no] = result
    umi_status.append({"group_no": group_no, "status": "estimated", "reason": ""})
    display(result["summary"])
    result["summary"].to_csv(
        f"{OUT_DIR}/{group_no}_TRA_TRB_clone_UMI_corr_summary.csv", index=False
    )
    result["pair_df"].to_csv(
        f"{OUT_DIR}/{group_no}_TRA_TRB_clone_UMI_corr_pairwise.csv", index=False
    )

pd.DataFrame(umi_status).to_csv(f"{OUT_DIR}/TRA_TRB_clone_UMI_status.csv", index=False)
if umi_corr_results:
    box_pearson = collect_boxplot_table_from_corr_results(
        umi_corr_results, corr_type="pearson"
    )
    box_pearson.to_csv(
        f"{OUT_DIR}/TRA_TRB_clone_UMI_pearson_pairwise_table.csv",
        index=False,
    )

print("TRA/TRB UMI-correlation tables saved; auxiliary pairwise figures are omitted.")


## Within-group mouse overlap analysis


In [ ]:
print("Within-group mouse Venn panels are omitted from this study output.")


## Structural robustness checks

The combined analysis includes repeated 15,000-UMI rarefaction and balanced-mouse background subsampling.


In [ ]:
def _tables_for_chain(chain, raw=False):
    suffix = chain.lower()
    prefix = "raw_ct" if raw else "ct"
    return {
        group: globals()[f"{prefix}_{group}_aaV_{suffix}"]
        for group in ("g1", "g2", "g3", "g4", "g5", "g6")
    }


def _save_stability_figure(figure, filename):
    plt.figure(figure.number)
    plt.savefig(FIGURE_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


if STRATUM in REPRESENTATIVE_SEED_STRATA and RUN_MODE == "standard":
    for chain in ("TRA", "TRB"):
        write_stability(
            _tables_for_chain(chain, raw=True), _tables_for_chain(chain), metadata,
            DOWNSAMPLE_DEPTH, ACTIVE_STRUCTURAL_SEED, chain, RESULT_DIR,
            _save_stability_figure, reruns=TRA_RAREFACTIONS if chain == "TRA" else None,
        )


## Differential clonotype abundance and V-segment prioritization

The prespecified inferential contrast compares g1 with the pooled g5+g6 allogeneic
reference. edgeR retains independent sample or mouse-level analysis units. Fisher's
exact test provides a separate pooled-count sensitivity analysis and is not treated as
an independent biological replicate model.

In [ ]:
from src.differential_abundance import (
    count_result_path, prepare_counts, fit_fisher, fit_edger, rank_v_segments,
)


if RUN_MODE == "spike-in":
    differential_chains = (SPIKE_CHAIN,)
else:
    differential_chains = ("TRA", "TRB") if STRATUM == "all_combined" else ("TRA",)
count_inputs = {}
for chain in differential_chains:
    counts, sample_metadata = prepare_counts(_tables_for_chain(chain, raw=True), metadata, POOL_WITHIN_MOUSE)
    count_inputs[chain] = counts, sample_metadata
    counts.to_csv(RESULT_DIR / f"{chain.lower()}_aav_count_matrix.csv")
    sample_metadata.to_csv(RESULT_DIR / f"{chain.lower()}_analysis_units.csv")
    display(sample_metadata.groupby(["group_no", "contrast_group"], observed=True).size())
    print(f"{chain} differential count matrix: {counts.shape}")

In [ ]:
fisher_by_chain = {}
for chain, (counts, sample_metadata) in count_inputs.items():
    fisher = fit_fisher(counts, sample_metadata, N_WORKERS)
    fisher_by_chain[chain] = fisher
    fisher.to_csv(count_result_path(RESULT_DIR, chain, "fisher_aav_results.csv"), index=False)
    display(fisher.sort_values("Fisher_FDR_g1").head(20))

In [ ]:
import rpy2.robjects as ro


edger_by_chain = {}
edger_status_by_chain = {}
for chain, (counts, sample_metadata) in count_inputs.items():
    edger, status = fit_edger(counts, sample_metadata)
    edger_by_chain[chain] = edger
    edger_status_by_chain[chain] = status
    edger.to_csv(count_result_path(RESULT_DIR, chain, "edger_aav_results.csv"), index=False)
    print(f"{chain}: {status}")
    display(edger.head())

(RESULT_DIR / "R_sessionInfo.txt").write_text(
    "\n".join(str(line) for line in ro.r("capture.output(sessionInfo())")) + "\n"
)

In [ ]:
remaining_by_chain = {"TRA": g1_remaining_aaV_tra, "TRB": g1_remaining_aaV_trb}
retention_by_chain = {"TRA": cmp_g1_tra_all, "TRB": cmp_g1_all}
if RUN_MODE == "spike-in" and SPIKE_CASE == "baseline":
    pd.DataFrame(sorted(remaining_by_chain[SPIKE_CHAIN]), columns=["cdr3aa", "v"]).to_csv(
        RESULT_DIR / "spike_baseline_remainder.csv", index=False,
    )

In [ ]:
ranking_by_chain = {}
distinctive_by_chain = {}
for chain in differential_chains:
    ranking = rank_v_segments(
        remaining_by_chain[chain], edger_by_chain[chain], fisher_by_chain[chain], STRATUM, STRATUM_LABEL,
    )
    prefix = chain.lower() + "v"
    ranking.to_csv(RESULT_DIR / f"{prefix}_ranking.csv", index=False)
    ranking_by_chain[chain] = ranking

    distinctive = ranking.loc[
        ranking["evidence_class"].ne("tested without positive enrichment evidence")
    ].head(20).copy()
    if distinctive.empty:
        distinctive = ranking.head(10).copy()
        distinctive["evidence_class"] = "descriptive ranking; no positive inferential evidence"
    distinctive.to_csv(RESULT_DIR / f"distinctive_{prefix}.csv", index=False)
    distinctive_by_chain[chain] = distinctive
    display(distinctive)

report_chain = SPIKE_CHAIN if RUN_MODE == "spike-in" else "TRA"
trav_ranking = ranking_by_chain[report_chain]
distinctive_trav = distinctive_by_chain[report_chain]
EDGER_STATUS = edger_status_by_chain[report_chain]

In [ ]:
for chain in differential_chains:
    ACTIVE_CHAIN = chain
    comparison = fisher_by_chain[chain].merge(edger_by_chain[chain], on="feature_id", how="inner")
    comparison.to_csv(count_result_path(RESULT_DIR, chain, "fisher_edger_feature_comparison.csv"), index=False)
    edger_figure = plot_edger(ranking_by_chain[chain], chain, FIGURE_STRATUM_SUFFIX)
    for ax, panel in zip(edger_figure.axes, ("mean_effect", "significant_support"), strict=True):
        _save_figure_png(edger_figure, f"{chain}_g1_edger_{panel}.png",
                         origin="surplus", bbox_inches=panel_bbox(ax))
    _render_article(edger_figure, chain, "edger")
    _render_article(plot_fisher(comparison, chain, FIGURE_STRATUM_SUFFIX), chain, "fisher")
    _render_article(plot_fisher_v(ranking_by_chain[chain], chain, FIGURE_STRATUM_SUFFIX), chain, "fisher_v")

In [ ]:
def _subset_tables(tables, sample_ids):
    selected = {}
    for group, table in tables.items():
        subset = table.loc[:, table.columns.isin(sample_ids)]
        selected[group] = subset.loc[subset.sum(axis=1).gt(0)]
    return selected


def _subset_profile(subset_tables):
    comparison, _, _ = compare_full_vs_remaining_after_subtractions(
        subset_tables["g1"], [subset_tables[group] for group in ("g2", "g4", "g5", "g6")], "g1",
    )
    if comparison.empty:
        return comparison.assign(score_s=0.0, region_entropy=0.0)
    intersection = make_clone_intersection_table_venn3(
        subset_tables["g1"], subset_tables["g5"], subset_tables["g6"],
    )
    profile, _ = make_v_profile(comparison, intersection, "g1", "freq_vs_g1")
    return profile


if STRATUM == "all_combined" and RUN_MODE == "standard":
    for chain in ("TRA", "TRB"):
        ACTIVE_CHAIN = chain
        raw = _tables_for_chain(chain, raw=True)
        structural = _tables_for_chain(chain)
        depth = depth_check(combine_count_tables({group: raw[group] for group in ("g1", "g2")}),
                            _stable_sample_seed)
        depth.to_csv(RESULT_DIR / f"{chain.lower()}_depth_check.csv", index=False)
        _render_article(plot_depth_check(depth, chain, DOWNSAMPLE_DEPTH), chain, "depth")

        combined = combine_count_tables({group: structural[group] for group in ("g1", "g2")})
        points = mds_points(combined, metadata.loc[metadata["chain"].eq(chain)])
        points[["MDS1", "MDS2"]].to_csv(RESULT_DIR / f"{chain.lower()}_mds_coordinates.csv")
        _render_article(plot_mds(points, chain, DOWNSAMPLE_DEPTH), chain, "mds")

        profiles = {}
        thresholds_by_subset = {}
        for subset in ("CD8+", "CD4+"):
            ids = metadata.loc[
                metadata["chain"].eq(chain) & metadata["_cell_subset"].eq(subset[:3].lower()), "sample_id"
            ]
            tables = _subset_tables(structural, ids)
            profiles[subset] = _subset_profile(tables)
            thresholds_by_subset[subset] = _plot_thresholds(chain, subset, tables, cohort_tables=structural)
            profiles[subset].to_csv(RESULT_DIR / f"{chain.lower()}_{subset[:3].lower()}_v_profile.csv", index=False)
        _render_article(
            plot_subsets(profiles, chain, thresholds_by_subset=thresholds_by_subset), chain, "cd4_cd8",
        )

In [ ]:
top_rows = distinctive_trav.head(5)
candidate_text = (
    "; ".join(
        f"{row.v_gene} [{row.evidence_class}]" for row in top_rows.itertuples()
    )
    if len(top_rows)
    else "none"
)
conclusion_lines = [
    f"# Stratum-specific conclusion: {STRATUM_LABEL}",
    "",
    f"- Selected samples and analysis units are documented in `{RESULT_DIR.name}/selected_samples_by_chain_and_group.csv`.",
    f"- edgeR status: {EDGER_STATUS}",
    f"- Highest-ranked {report_chain} V segments: {candidate_text}.",
    "- Evidence classes distinguish concordant inference, single-method support, and exact-set descriptive ranking.",
    "- Absence of FDR-supported features is retained as a null result and is not converted into evidence of equivalence.",
]
(RESULT_DIR / "conclusion.md").write_text("\n".join(conclusion_lines) + "\n", encoding="utf-8")

print("\n".join(conclusion_lines))

In [ ]:
_persist_open_figures(origin="final-cell")

manifest = pd.DataFrame(PLOT_RECORDS)
if manifest.empty:
    raise RuntimeError("No figure was persisted; the notebook output is incomplete.")
manifest["png"] = manifest["png"].map(lambda value: str(Path(value).relative_to(REPOSITORY_ROOT)))

figure_names = {Path(path).name for path in manifest["png"]}
required_localization = {
    "TRA_g1_v_segment_localization.png",
    "TRB_g1_v_segment_localization.png",
}
missing_localization = required_localization - figure_names
if missing_localization:
    raise RuntimeError(
        "The figure contract requires TRA and TRB V-segment localization figures; "
        f"missing: {sorted(missing_localization)}"
    )

if not manifest["chain"].isin(["TRA", "TRB"]).all():
    unassigned = manifest.loc[~manifest["chain"].isin(["TRA", "TRB"]), "png"].tolist()
    raise RuntimeError(f"Every publication figure must belong to TRA or TRB: {unassigned[:5]}")

for chain in ("TRA", "TRB"):
    chain_rows = manifest[manifest["chain"].eq(chain)]
    if chain_rows.empty:
        raise RuntimeError(f"No {chain} figures were persisted for {STRATUM}.")
    bad_paths = [
        path for path in chain_rows["png"]
        if chain not in Path(path).parts
    ]
    if bad_paths:
        raise RuntimeError(f"{chain} figures are not stored in the {chain} directory: {bad_paths[:5]}")

manifest.to_csv(RESULT_DIR / "figure_manifest.csv", index=False)
if V_THRESHOLD_RECORDS:
    pd.DataFrame(V_THRESHOLD_RECORDS).to_csv(RESULT_DIR / "v_thresholds.csv", index=False)

missing_figure_files = [
    record["png"] for record in PLOT_RECORDS if not Path(record["png"]).is_file()
]
if missing_figure_files:
    raise RuntimeError(f"Missing persisted figure files: {missing_figure_files[:10]}")

run_summary = {
    "stratum": STRATUM,
    "mode": RUN_MODE,
    "v_threshold_mode": V_THRESHOLD_MODE,
    "spike_case": SPIKE_CASE if RUN_MODE == "spike-in" else None,
    "stratum_label": STRATUM_LABEL,
    "pool_within_mouse": POOL_WITHIN_MOUSE,
    "structural_downsample_umi": DOWNSAMPLE_DEPTH,
    "structural_downsample_seed": int(ACTIVE_STRUCTURAL_SEED),
    "representative_seed_selected_from_100_reruns": bool(
        STRATUM in REPRESENTATIVE_SEED_STRATA and RUN_MODE == "standard"
    ),
    "stability_uses_spike_in": False,
    "functional_only": True,
    "n_selected_samples": int(metadata["sample_id"].nunique()),
    "n_selected_mice": int(metadata["mouse_id"].nunique()),
    "n_figures": int(len(manifest)),
    "n_ranked_trav": int(len(ranking_by_chain.get("TRA", ()))),
    "n_distinctive_trav": int(len(distinctive_by_chain.get("TRA", ()))),
    "spike_chain": SPIKE_CHAIN if RUN_MODE == "spike-in" else None,
    "edger_status": EDGER_STATUS,
    "edger_status_by_chain": edger_status_by_chain,
}
(RESULT_DIR / "run_summary.json").write_text(
    json.dumps(run_summary, indent=2) + "\n", encoding="utf-8"
)

if RUN_MODE == "standard" and os.environ.get("MICE_TCR_FINALIZE_SUMMARY", "0") == "1":
    summary_tables = []
    for stratum_name in STRATUM_CONFIG:
        table_path = RESULT_ROOT / stratum_name / "distinctive_trav.csv"
        if table_path.is_file():
            summary_tables.append(pd.read_csv(table_path))
    if summary_tables:
        all_strata_summary = pd.concat(summary_tables, ignore_index=True)
        if all_strata_summary.empty:
            raise RuntimeError(
                "The consolidated TRAV table is empty; inspect the nine stratum "
                "executions before interpreting a biological null result."
            )
        all_strata_summary.to_csv(
            RESULT_ROOT / "all_strata_distinctive_trav_summary.csv", index=False
        )
        display(all_strata_summary)

print(f"Persisted figures: {len(PLOT_RECORDS)}")
print(f"Executed-notebook result set completed for {STRATUM}.")

if SPIKE_SELECTION is not None:
    write_case_metrics(
        RESULT_DIR, SPIKE_SELECTION, SPIKE_FRACTION, _tables_for_chain(SPIKE_CHAIN, raw=True)["g1"],
        remaining_by_chain[SPIKE_CHAIN], retention_by_chain[SPIKE_CHAIN],
        edger_by_chain[SPIKE_CHAIN], ranking_by_chain[SPIKE_CHAIN],
    )
if EDGER_STATUS != "edgeR quasi-likelihood model fitted successfully.":
    raise RuntimeError(EDGER_STATUS)